# 머신러닝 파일별 보강1 · 4-2 KNN · 거리 · 스케일링 · K · 차원의 저주

**한 줄 요약**

- '가까운 애들끼리 같은 라벨'은 어떻게 동작하는가?
- KNN에서 좋은 거리란 무엇인가를 스케일링 · K · weights · 피처
- 차원의 저주로 확인하고 test는 마지막에 한 번

- **원본** · `6.머닝러신/04_분류모델/4-2_KNN.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/05_머신러닝` 머신러닝-2 (스케일링 · Pipeline) · 파일별 2 · 3의 펭귄 KNN 0.7801과 이어짐

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. KNN이 '가까운 K개 이웃의 다수결'로 분류하는 과정을 거리 계산으로 직접 보인다
2. 단위가 다르면 스케일링이 왜 필요한지, 한 번의 split 결과를 왜 믿으면 안 되는지 말한다
3. K가 작을수록 과적합 · 클수록 과소적합이 경계와 train-CV 격차에서 어떻게 보이는지 설명한다
4. `weights='distance'`의 train 1.0 · GridSearchCV 공동 1위를 올바르게 해석한다
5. 정답과 무관한 차원이 늘면 KNN이 무너지는 차원의 저주를 실험으로 보인다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **비슷한 고객 · 상품을 찾아 추천할 때** · 거리 · 이웃 다수결 (1장)
- **단위가 다른 특성이 섞인 표에 KNN을 쓸 때** · 스케일링 · Pipeline (2장)
- **K를 정해야 할 때** · train-CV 격차 · GridSearchCV (3 · 5장)
- **특성이 수백 개인 데이터에 KNN을 쓰려 할 때** · 차원의 저주 · 무관한 특성 (6장)

## 학습 지도

<img src="images/pfm8_map.png" alt="4-2 KNN 학습 지도" width="763">

**예상 학습 시간** · 개념 1시간 + 실습 1시간 (전체 실행 1분 이내 · GPU 필요 없음)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>Iris · KNN 준비 · 원본 목표</td></tr>
<tr><td>1</td><td>KNN 원리</td><td>KNN은 새 데이터를 어떻게 분류하고, 혼동행렬로는 무엇을 보는가?</td></tr>
<tr><td>2</td><td>스케일링</td><td>단위가 다르면 거리는 어떻게 왜곡되고, Iris에서 본 '스케일링 후 하락'은 진짜인가?</td></tr>
<tr><td>3</td><td>K와 경계</td><td>K를 바꾸면 train과 CV 정확도, 그리고 경계 모양은 어떻게 변하는가?</td></tr>
<tr><td>4</td><td>피처와 중요도</td><td>중요한 피처만 남기면 좋아지는가? 계수가 없는 KNN도 피처 중요도를 볼 수 있는가?</td></tr>
<tr><td>5</td><td>GridSearchCV</td><td>K와 weights를 GridSearchCV로 고르면 유일한 승자가 나오는가? distance의 train 1.0은 무슨 뜻인가?</td></tr>
<tr><td>6</td><td>차원의 저주</td><td>피처를 계속 늘리면 좋아지는가? 가까운 것과 먼 것의 차이는 어떻게 되는가?</td></tr>
<tr><td>7</td><td>종합 실습</td><td>dev에서 고른 Iris KNN으로 test를 한 번 보고, 앞 회차 펭귄 KNN 0.7801을 오늘 배운 것으로 고치면?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

Iris는 scikit-learn 내장 데이터 · 7장의 펭귄만 원본 폴더 CSV를 읽음. **셀은 위에서부터 차례로 실행**.

*항목 → 설정*

- **라이브러리** · numpy · pandas · scikit-learn · scipy(6-1)
- **seed** · 분할 · CV `random_state=42` (원본과 같음)
- **데이터** · Iris 150 · 가짜 단위 데이터 200 · 난수 · 펭귄 342마리(7장)
- **원본 그림** · 슬라이드 스크린샷 9장 → 💡 · 비교 표의 글로

**실행 흐름**

1. 모델 · 평가 도구 불러오기 (그림은 images/의 정적 그림)
2. 스케일러 + KNN Pipeline 함수 만들기
3. Iris · CV 계약 준비

In [1]:
# 준비: Iris · KNN · 스케일러 (원본의 plotly 그림은 images/의 정적 그림으로)
# 왜 · GitHub에서도 그림이 보이도록 정적 그림을 쓰고 도구를 한곳에 모으기 위해
import numpy as np                                         # 배열 · 난수
import pandas as pd                                        # 표
from sklearn.datasets import load_iris                     # Iris
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # 지표
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split  # 분할 · CV
from sklearn.neighbors import KNeighborsClassifier as KNN  # KNN (짧은 이름)
from sklearn.pipeline import Pipeline                      # 전처리 + 모델
from sklearn.preprocessing import StandardScaler           # 스케일링


# 왜 · KNN은 거리 기반이라 스케일링을 항상 같이 묶어 실수를 막기 위해
def knn_pipe(k=5, weights="uniform"):          # 스케일러 + KNN Pipeline (원본에서 반복되던 코드)
    knn = KNN(n_neighbors=k, weights=weights)  # 이웃 수 · 투표 방식
    return Pipeline([("scaler", StandardScaler()), ("knn", knn)])  # train에만 fit되는 scaler


# 왜 · 모든 실험이 같은 데이터 · 같은 CV 계약을 쓰도록 한 번만 준비하기 위해
iris = load_iris()                                               # 150송이 · 4특성 (전부 cm)
X, y = iris.data, iris.target                                    # 문제 · 정답
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 오늘의 CV
print("Iris:", X.shape, "· 품종:", iris.target_names.tolist())   # 확인

Iris: (150, 4) · 품종: ['setosa', 'versicolor', 'virginica']


### 원본 목표 · 척추 질문

- **핵심 질문** · "가까운 애들끼리 같은 라벨"은 어떻게 동작하는가?
- 목표 · KNN 원리 · K 값의 영향 · 스케일링 · Decision Boundary · Pipeline + GridSearchCV
- 앞 회차 펭귄 KNN은 0.7801이었음 → 오늘 그 안을 열어 왜 0.78이었는지 확인

*오늘 다룰 것 → 사실은 이 질문 (**KNN에서 좋은 거리란 무엇인가?**)*

- **스케일링** · 단위 때문에 거리가 **왜곡**되지는 않았나?
- **피처 선택** · 쓸모없는 정보가 거리를 **오염**시키지는 않나?
- **차원의 저주** · 좌표가 너무 많아 거리 자체의 **의미가 약해**지지는 않나?
- **K와 weights** · 찾은 이웃의 의견을 **어떻게 모을** 것인가?

**원본 힌트**

- 로지스틱: 점수가 높으면 양성 · KNN: 가까운 이웃이 양성이면 나도 양성
- 로지스틱은 가중치(w, b)를 학습 · KNN은 학습 없이 데이터를 저장하고 예측할 때 거리 계산

---
## 1. KNN 원리

> **이 장의 흐름**
>
> - Iris에 KNN(K=5)을 그대로 써 보고 3 × 3 혼동행렬 확인
> - 그다음 한 송이를 골라 거리를 직접 계산해 이웃 5개의 다수결을 따라가 봄

<img src="images/pfm8_vote.png" alt="새 데이터의 가까운 이웃 K개 다수결" width="378">

**파이프라인 위치** · ④ 모델 — 거리 기반 분류

### 1-1. `A` Iris KNN 기본 · `KNeighborsClassifier` · `metric="minkowski", p=2` <sub>원본 셀 3 ~ 6</sub>

> 🎯 **KNN(K=5, 유클리드 거리)로 Iris test를 몇 개 맞히는가?**

💡 **쉽게 말하면**

- 새 데이터가 오면 학습 데이터에서 **가까운 K개**를 찾고, 그 라벨의 **다수결**.
- 학습 단계에서는 데이터를 저장만 함.

**실행 흐름**

1. 8 : 2 분할
2. KNN(K=5, p=2) fit
3. test 정확도 · 클래스별 표

**먼저 예측** · test 30송이 중 몇 개를 맞힐까? — 답은 코드 실행 결과 아래에

In [2]:
# 왜 · 기본 KNN의 학습 · 예측 · 채점 흐름을 먼저 확인하기 위해
X_train, X_test, y_train, y_test = train_test_split(            # 원본 셀 6
    X, y, test_size=0.2, stratify=y, random_state=42)
model = KNN(n_neighbors=5, metric="minkowski", p=2)             # p=2 → 유클리드 거리
model.fit(X_train, y_train)                                     # 학습 = 데이터 저장
y_pred = model.predict(X_test)                                  # 이웃 5개 다수결
print(f"[Test Accuracy] {accuracy_score(y_test, y_pred):.3f}")  # 원본 형식
print(classification_report(y_test, y_pred, target_names=iris.target_names, digits=3))  # 클래스별

[Test Accuracy] 1.000
              precision    recall  f1-score   support

      setosa      1.000     1.000     1.000        10
  versicolor      1.000     1.000     1.000        10
   virginica      1.000     1.000     1.000        10

    accuracy                          1.000        30
   macro avg      1.000     1.000     1.000        30
weighted avg      1.000     1.000     1.000        30



**결과** · 예측 정답은 30개 전부 (1.000)

- `[Test Accuracy] 1.000` — Iris는 분리가 쉬운 교육용 데이터
- `accuracy                          1.000        30` — 세 품종 모두 정밀도 · 재현율 1.000

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>K 작음 (1 ~ 3)</td><td>경계가 복잡 · 노이즈에 민감</td><td>과적합 위험</td></tr>
<tr><td>K 큼 (15 ~)</td><td>경계가 단순 · 세부 패턴 놓침</td><td>과소적합 위험</td></tr>
<tr><td>K 적당</td><td>균형</td><td>교차검증으로 찾기 (3 · 5장)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 4개 피처가 전부 cm라 스케일링 없이도 잘 나옴 → 단위가 섞이면 다름 (2장)

> ✅ **한 줄 정리** · KNN은 학습 때 데이터를 저장하고, 예측 때 가까운 K개 이웃의 다수결로 라벨을 정한다.

**딥러닝 코드에서는**

- 임베딩 공간의 최근접 이웃 검색 (얼굴 인식 · 추천 · RAG)도 같은 생각

### 1-2. `B` 3 × 3 혼동행렬 · `confusion_matrix` <sub>원본 셀 7 ~ 9</sub>

> 🎯 **클래스가 3개면 혼동행렬은 어떻게 읽는가?**

💡 **쉽게 말하면**

- 3-2는 이진(2 × 2) · 여기는 **3 × 3**.
- 대각선 = 맞힌 수 · 대각선 밖 = 어떤 클래스끼리 헷갈렸나.

**실행 흐름**

1. 혼동행렬
2. 행 = 실제 · 열 = 예측
3. 대각선 합 = 맞힌 수

**먼저 예측** · 대각선 밖에 숫자가 있을까? — 답은 코드 실행 결과 아래에

In [3]:
# 왜 · 어느 품종끼리 헷갈리는지 혼동행렬로 확인하기 위해
cm = confusion_matrix(y_test, y_pred)                      # 행 = 실제, 열 = 예측
print(pd.DataFrame(cm, index=iris.target_names, columns=iris.target_names).to_string())  # 표
print("대각선 합(맞힌 수):", np.trace(cm), "/", cm.sum())  # 30 / 30

            setosa  versicolor  virginica
setosa          10           0          0
versicolor       0          10          0
virginica        0           0         10
대각선 합(맞힌 수): 30 / 30


**결과** · 예측 정답은 없음 — 대각선만 10 · 10 · 10

- `setosa          10           0          0` — setosa 10 · 헷갈림 없음
- `대각선 합(맞힌 수): 30 / 30` — 정확도 1.000과 같은 뜻

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>2 × 2 (이진)</td><td>TN · FP / FN · TP</td><td>파일별 4</td></tr>
<tr><td>3 × 3 (다중)</td><td>대각선 = 정답 · 나머지 = 헷갈린 쌍</td><td>어떤 품종끼리 헷갈리나</td></tr>
<tr><td>정확도</td><td>대각선 합 ÷ 전체</td><td>한 숫자 요약</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 정확도만 보면 어떤 클래스끼리 헷갈리는지 안 보임 → 혼동행렬을 함께

> ✅ **한 줄 정리** · 다중 분류 혼동행렬은 대각선이 정답 수, 나머지 칸이 어떤 클래스끼리 헷갈렸는지를 보여 준다.

**딥러닝 코드에서는**

- 10 × 10 혼동행렬로 MNIST의 4 ↔ 9 같은 헷갈림 쌍 찾기

### 1-3. `A` 거리를 직접 계산해 이웃 찾기 · `np.linalg.norm` · `kneighbors` <sub>원본 셀 3 · 4</sub>

> 🎯 **test 첫 송이의 가까운 이웃 5개는 누구이고, 다수결은 sklearn과 같은가?**

💡 **쉽게 말하면**

- 유클리드 거리 = √(Σ 차이²) (피타고라스).
- 거리를 재니 **단위 · 범위가 다르면 큰 값이 거리를 지배**.

**실행 흐름**

1. 모든 train과의 거리
2. 가장 가까운 5개
3. 라벨 다수결 · sklearn kneighbors와 비교

**먼저 예측** · 손으로 찾은 이웃 5개와 sklearn의 이웃 5개는 같을까? — 답은 코드 실행 결과 아래에

In [4]:
# 왜 · KNN이 가장 가까운 이웃 5개의 다수결로 예측한다는 것을 직접 계산해 보기 위해
star = X_test[0]                                    # 새 꽃 한 송이
dists = np.linalg.norm(X_train - star, axis=1)      # train 120송이까지 거리
near = np.argsort(dists)[:5]                        # 가장 가까운 5개 번호
print("새 꽃 측정값:", star.tolist(), "· 실제 품종:", iris.target_names[y_test[0]])  # 새 꽃
print("이웃 거리:", dists[near].round(3).tolist())  # 가까운 순
print("이웃 라벨:", y_train[near].tolist(), "→ 다수결", np.bincount(y_train[near]).argmax())  # 투표
_, sk_idx = model.kneighbors([star])                # sklearn이 찾은 이웃
print("sklearn 이웃과 같음:", sorted(sk_idx[0].tolist()) == sorted(near.tolist()))  # 일치 확인

새 꽃 측정값: [4.4, 3.0, 1.3, 0.2] · 실제 품종: setosa
이웃 거리: [0.141, 0.245, 0.3, 0.3, 0.361]
이웃 라벨: [0, 0, 0, 0, 0] → 다수결 0
sklearn 이웃과 같음: True


**결과** · 예측 정답은 같음

- `새 꽃 측정값: [4.4, 3.0, 1.3, 0.2] · 실제 품종: setosa` — 새 꽃
- `이웃 라벨: [0, 0, 0, 0, 0] → 다수결 0` — 이웃 5개가 모두 setosa
- `sklearn 이웃과 같음: True` — KNN 예측 = 거리 계산 + 다수결

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>유클리드 (p=2)</td><td>√(Σ 차이²)</td><td>기본</td></tr>
<tr><td>맨해튼 (p=1)</td><td>Σ</td><td>차이</td></tr>
<tr><td>코사인</td><td>방향만</td><td>텍스트 · 임베딩 (파일별 3)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 거리는 모든 피처를 그대로 더함 → 단위가 큰 피처가 거리를 지배 (2-1)

> ✅ **한 줄 정리** · KNN 예측은 모든 train과의 거리를 재서 가장 가까운 K개를 고르고 그 라벨을 다수결하는 것이다.

**딥러닝 코드에서는**

- `torch.cdist` · FAISS로 임베딩 사이 거리를 대량으로 계산

---
## 2. 스케일링

> **이 장의 흐름**
>
> - 0 ~ 1과 0 ~ 1000이 섞인 데이터로 스케일링 전후 → Iris Pipeline 한 번 분할
> - 그다음 split 20번 · 5-Fold로 '하락'을 다시 확인

<img src="images/pfm8_scale.png" alt="0 ~ 1 vs 0 ~ 1000 · 스케일링 전후" width="912">

**파이프라인 위치** · ③ 전처리 — 거리의 기준

### 2-1. `A` 0 ~ 1 vs 0 ~ 1000 · `StandardScaler` · `fit_transform` <sub>원본 셀 10 ~ 12</sub>

> 🎯 **범위가 1000배 다른 두 특성으로 KNN을 하면 스케일링 전후 정확도는?**

💡 **쉽게 말하면**

- feature2(0 ~ 1000)가 거리를 지배 → feature1(0 ~ 1)은 무시됨.
- 스케일링으로 범위를 맞춰야 공정한 거리.

**실행 흐름**

1. 두 특성 200개 · 정답은 둘 다 사용
2. 스케일링 없이 KNN
3. train에 fit한 scaler로 변환 후 KNN

**먼저 예측** · 스케일링 전후 정확도 차이는 얼마나 클까? — 답은 코드 실행 결과 아래에

In [5]:
# 왜 · 단위가 큰 특성이 거리를 지배하는 문제를 가짜 데이터로 재현하기 위해
rng_scale = np.random.RandomState(42)                                        # 원본 셀 11
X_demo = np.c_[rng_scale.rand(200), rng_scale.rand(200) * 1000]              # 0 ~ 1 · 0 ~ 1000
y_demo = (X_demo[:, 0] + X_demo[:, 1] / 1000 > 1.0).astype(int)  # 두 특성을 같은 비중으로 쓰는 규칙
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(                               # 7 : 3
    X_demo, y_demo, test_size=0.3, stratify=y_demo, random_state=42)
acc_raw = accuracy_score(yd_te, KNN(5).fit(Xd_tr, yd_tr).predict(Xd_te))     # 그대로
sc = StandardScaler().fit(Xd_tr)                                             # train에만 fit
Xs_tr, Xs_te = sc.transform(Xd_tr), sc.transform(Xd_te)                      # 같은 기준으로 변환
acc_scaled = accuracy_score(yd_te, KNN(5).fit(Xs_tr, yd_tr).predict(Xs_te))  # 스케일링 후
print(f"스케일링 없음: {acc_raw:.6f} · StandardScaler: {acc_scaled:.6f}")    # 원본 셀 11의 표

스케일링 없음: 0.650000 · StandardScaler: 0.983333


**결과** · 예측 정답은 극적 — 0.65 → 0.98

- `스케일링 없음: 0.650000 · StandardScaler: 0.983333` — feature1이 무시되다 살아남

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>스케일링 없음</td><td>큰 범위 특성이 거리를 지배</td><td>단위가 같을 때만</td></tr>
<tr><td>StandardScaler</td><td>평균 0 · 표준편차 1</td><td><strong>거리 기반 모델의 기본</strong></td></tr>
<tr><td>Pipeline</td><td>scaler를 train에만 fit · 누수 방지</td><td>항상 (2-2)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - scaler는 train에만 fit하고 test는 transform만 (파일별 3)

> ✅ **한 줄 정리** · 범위가 큰 특성이 거리를 지배하므로, KNN에서는 스케일링이 공정한 거리 비교의 전제다.

**딥러닝 코드에서는**

- 입력 정규화 · 임베딩 정규화 — 거리를 쓰는 곳은 항상 크기부터 맞춤

### 2-2. `B` Iris · 한 번 분할의 '하락' · `knn_pipe` · `Pipeline` <sub>원본 셀 12 · 13</sub>

> 🎯 **Iris에 스케일링 Pipeline을 쓰면 정확도가 1.000에서 떨어지는가?**

💡 **쉽게 말하면**

- 한 번의 분할에서는 스케일링 후 **0.933**으로 내려감.
- 그런데 이 숫자를 믿어도 될까? (2-3)

**실행 흐름**

1. 같은 분할에서 KNN 그대로
2. Pipeline(Scaler + KNN)
3. 비교

**먼저 예측** · 스케일링하면 1.000보다 오를까 내릴까? — 답은 코드 실행 결과 아래에

In [6]:
# 왜 · 스케일링 유무에 따른 점수 차이를 같은 분할로 비교하기 위해
pipe_knn = knn_pipe(5)                                                             # Scaler + KNN
acc_raw = accuracy_score(y_test, KNN(5).fit(X_train, y_train).predict(X_test))     # 그대로
acc_pipe = accuracy_score(y_test, pipe_knn.fit(X_train, y_train).predict(X_test))  # 스케일링
print(f"KNN (스케일링 없음): {acc_raw:.6f} · Pipeline (Scaler + KNN): {acc_pipe:.6f}")  # 원본 셀 13

KNN (스케일링 없음): 1.000000 · Pipeline (Scaler + KNN): 0.933333


**결과** · 예측 정답은 내림 — 1.000 → 0.933

- `KNN (스케일링 없음): 1.000000 · Pipeline (Scaler + KNN): 0.933333` — test 30개에서 2개 차이

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>스케일링 전</td><td>범위가 큰 피처(petal length 등)가 거리를 주도</td><td>우연히 그 피처가 유용하면 높게 나옴</td></tr>
<tr><td>스케일링 후</td><td>모든 피처를 같은 척도로</td><td>판단 기준이 바뀜</td></tr>
<tr><td>한 번의 split</td><td>2개 차이 · 운이 섞임</td><td>결론 내리지 않기 (2-3)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 스케일링이 맞춰 주는 것은 **숫자의 범위**지 **정보의 유용성**이 아님
> - '숫자가 크다는 이유만으로 목소리가 커지는 문제'를 없애는 것

> ✅ **한 줄 정리** · 스케일링은 이웃 관계(판단 기준)를 바꾸므로 한 번의 분할에서 점수가 달라질 수 있지만, 그것만으로 좋고 나쁨을 정하지 않는다.

**딥러닝 코드에서는**

- 정규화 방식을 바꾸면 결과가 달라짐 → 한 번의 학습으로 판단하지 않기

### 2-3. `A` split 20번 · 5-Fold로 다시 · `for seed in range(20)` · `cross_val_score` <sub>원본 셀 14 ~ 17</sub>

> 🎯 **split을 20번 바꾸고 5-Fold로 재면 '스케일링 후 하락'이 그대로 나오는가?**

💡 **쉽게 말하면**

- **한 번의 split으로 결론 내지 않음** (3회차).
- 여러 번 · 교차검증으로 다시 잼.

**실행 흐름**

1. seed 0 ~ 19로 분할 20번
2. 둘 중 어느 쪽이 높았나 세기
3. 5-Fold 평균

**먼저 예측** · 교차검증에서도 스케일링 쪽이 낮을까? — 답은 코드 실행 결과 아래에

In [7]:
# 왜 · 한 번의 분할 운이 아닌 20번의 분할로 스케일링 효과를 확인하기 위해
plain, piped = [], []                                                   # 분할마다 정확도
for seed in range(20):                                                  # split 20번
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)  # 새 분할
    plain.append(KNN(5).fit(Xa, ya).score(Xb, yb))                      # 그대로
    piped.append(knn_pipe(5).fit(Xa, ya).score(Xb, yb))                 # 스케일링
plain, piped = np.array(plain), np.array(piped)                         # 배열로
win, tie, lose = (plain > piped).sum(), (plain == piped).sum(), (piped > plain).sum()  # 세 경우
# 원본 셀 15
print(f"[split 20번] 스케일링 없음이 높음 {win} · 동점 {tie} · 스케일링이 높음 {lose}")
s_plain = cross_val_score(KNN(5), X, y, cv=cv).mean()                   # 5-Fold
s_pipe = cross_val_score(knn_pipe(5), X, y, cv=cv).mean()               # 5-Fold
print(f"[5-Fold] 스케일링 없음 {s_plain:.4f} · 스케일링 {s_pipe:.4f}")  # 순서가 뒤집힘

[split 20번] 스케일링 없음이 높음 8 · 동점 11 · 스케일링이 높음 1
[5-Fold] 스케일링 없음 0.9667 · 스케일링 0.9733


**결과** · 예측 정답은 아님 — 5-Fold에서는 스케일링 쪽이 높음

- `[split 20번] 스케일링 없음이 높음 8 · 동점 11 · 스케일링이 높음 1` — 대부분 동점
- `[5-Fold] 스케일링 없음 0.9667 · 스케일링 0.9733` — 교차검증에서는 순서가 뒤집힘

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>한 번 split (2-2)</td><td>1.000 → 0.933 '하락'</td><td>random_state=42에서만 보이는 그림</td></tr>
<tr><td>split 20번</td><td>8 : 11 : 1</td><td>흔들림을 보여 줌</td></tr>
<tr><td>5-Fold CV</td><td>0.9667 &lt; 0.9733</td><td>비교의 기본</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 스케일링은 정확도를 올리는 마법이 아니라 **거리의 기준을 바꾸는 전처리**
> - Iris는 전부 cm라 효과가 작음 · 단위가 크게 다르면(2-1) 효과가 훨씬 크고 안정적

> ✅ **한 줄 정리** · 한 번의 split에서 본 '스케일링 후 하락'은 착시였고, 결과의 좋고 나쁨은 여러 번 · 교차검증으로 판단한다.

**딥러닝 코드에서는**

- seed 하나로 본 개선 · 하락을 seed 여러 개로 다시 확인

---
## 3. K와 경계

> **이 장의 흐름**
>
> - K = 1 ~ 15에서 train vs CV 정확도와 격차
> - 그다음 petal 2개로 K별 결정 경계 · test 정확도

<img src="images/pfm8_boundary.png" alt="petal 2차원 · K=1 · 5 · 15의 결정 경계" width="951">

**파이프라인 위치** · ④ 모델 — 복잡도

### 3-1. `A` K별 train vs CV · `n_neighbors` · `cross_val_score` <sub>원본 셀 18 ~ 20</sub>

> 🎯 **K = 1이면 train 정확도는 몇이고, train과 CV 중 어느 쪽이 K에 민감한가?**

💡 **쉽게 말하면**

- K = 1이면 train을 예측할 때 **자기 자신이 이웃** → train 1.000.
- 과적합은 train 점수가 아니라 **train-CV 격차**로 봄.

**실행 흐름**

1. K 후보 8개
2. train 정확도 · train 안 5-Fold CV
3. 격차 표 (test는 안 씀)

**먼저 예측** · K = 1의 train 정확도는? — 답은 코드 실행 결과 아래에

In [8]:
rows = []                                                               # K마다
# 왜 · K가 작으면 과적합, 크면 과소적합이 되는 흐름을 표로 보기 위해
for k in [1, 3, 5, 7, 9, 11, 13, 15]:                                   # 원본 셀 20의 후보
    pipe = knn_pipe(k).fit(X_train, y_train)                            # train에 fit
    tr = accuracy_score(y_train, pipe.predict(X_train))                 # 외운 문제
    cvs = cross_val_score(knn_pipe(k), X_train, y_train, cv=cv).mean()  # train 안 CV
    rows.append([k, tr, cvs, tr - cvs])                                 # 한 줄
k_df = pd.DataFrame(rows, columns=["K", "train 정확도", "CV 정확도", "격차 (train - CV)"])  # 표
print(k_df.round(4).to_string(index=False))                             # 원본 표

 K  train 정확도  CV 정확도  격차 (train - CV)
 1     1.0000  0.9417           0.0583
 3     0.9583  0.9667          -0.0083
 5     0.9750  0.9583           0.0167
 7     0.9750  0.9333           0.0417
 9     0.9583  0.9583           0.0000
11     0.9583  0.9500           0.0083
13     0.9583  0.9500           0.0083
15     0.9667  0.9250           0.0417


**결과** · 예측 정답은 1.000 — 자기 자신이 이웃

- `1        1.0000   0.9417           0.0583` — train 1.0 · 격차 가장 큼 (과적합)
- `3        0.9583   0.9667          -0.0083` — CV 최고 · 격차 거의 0
- `15        0.9667   0.9250           0.0417` — K가 커지면 CV가 다시 내려감 (과소적합 쪽)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>K = 1</td><td>자기 자신 · 가장 가까운 한 점</td><td>train 1.0 · 노이즈에 민감</td></tr>
<tr><td>K 적당</td><td>격차 작고 CV 높음</td><td>고를 후보</td></tr>
<tr><td>K 큼</td><td>멀리 있는 점까지 투표</td><td>경계가 단순 · 세부 패턴 놓침</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 표의 값은 test 30개를 쓰지 않은 train 120개 안의 CV · test는 7장에서 한 번

> ✅ **한 줄 정리** · K가 작으면 train은 높고 CV와의 격차가 크며(과적합), K가 너무 크면 CV가 다시 떨어지므로 격차와 CV로 K를 고른다.

**딥러닝 코드에서는**

- 모델 크기를 바꾸며 train · val 곡선을 함께 보는 것과 같음

### 3-2. `B` 결정 경계 · petal 2개 · `predict(grid)` · `score` <sub>원본 셀 21 ~ 23</sub>

> 🎯 **K = 1과 K = 15의 경계는 어떻게 다르고, test 정확도는?**

💡 **쉽게 말하면**

- K가 작으면 경계가 **울퉁불퉁** (점 하나에도 휘어짐).
- K가 크면 **매끈** (세부를 무시).

**실행 흐름**

1. petal 2개 · 7 : 3 분할
2. K 6개 Pipeline
3. 격자 예측(그림) · test 정확도

**먼저 예측** · K가 커지면 test 정확도도 계속 오를까? — 답은 코드 실행 결과 아래에

In [9]:
# 왜 · 특성 두 개만 써서 K에 따른 경계 변화를 그림으로 보기 위해
X_2d = X[:, 2:4]                    # petal length · width
# 원본 셀 22
X_tr, X_te, y_tr, y_te = train_test_split(X_2d, y, test_size=0.3, stratify=y, random_state=42)
acc_map = {}                        # K마다 test 정확도
for k in [1, 3, 5, 7, 9, 15]:       # 원본 드롭다운 후보
    acc_map[k] = round(knn_pipe(k).fit(X_tr, y_tr).score(X_te, y_te), 3)  # test 45개
print("K별 test 정확도:", acc_map)  # 경계 그림(3장)과 함께 읽기

K별 test 정확도:

 {1: 1.0, 3: 0.933, 5: 0.911, 7: 0.911, 9: 0.911, 15: 0.911}


**결과** · 예측 정답은 아님 — 이 test에서는 K = 1이 1.0으로 가장 높음

- `K별 test 정확도: {1: 1.0, 3: 0.933, 5: 0.911, 7: 0.911, 9: 0.911, 15: 0.911}` — test 45개에서는 K=1이 최고 · 하지만 test로 K를 고르면 안 됨 (선택은 3-1의 CV)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>울퉁불퉁한 경계 (K = 1)</td><td>train 점 하나하나를 감쌈</td><td>과적합</td></tr>
<tr><td>매끈한 경계 (K = 15)</td><td>큰 흐름만</td><td>과소적합 위험</td></tr>
<tr><td>경계 그림 + 점수 표</td><td>그림의 모양과 3-1 격차를 연결</td><td>설명의 기본</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - petal 2개만 쓴 2차원 그림 · 4차원 전체에서의 경계와는 다름
> - 원본은 드롭다운으로 K를 바꿔 보는 그림 → 여기서는 K=1 · 5 · 15 세 장으로

> ✅ **한 줄 정리** · K가 작으면 경계가 점마다 휘어 과적합되고, 크면 매끈해져 과소적합 쪽으로 가며, 그 모습이 3-1의 격차 표와 같은 이야기다.

**딥러닝 코드에서는**

- 결정 경계 시각화는 2차원 임베딩(t-SNE · UMAP) 위에서 자주 함

---
## 4. 피처와 중요도

> **이 장의 흐름**
>
> - petal 2개 vs 4개 전체를 같은 CV로 → 차이를 표준편차와 비교
> - 그다음 permutation importance로 KNN의 피처 중요도를 사후에 측정

<img src="images/pfm8_perm.png" alt="KNN permutation importance · petal ≫ sepal" width="595">

**파이프라인 위치** · ③ 피처 선택 · ⑥ 해석

### 4-1. `B` petal 2개 vs 전체 4개 · `X[:, 2:4]` · `cross_val_score` <sub>원본 셀 24 ~ 26</sub>

> 🎯 **petal 2개만 쓰면 4개 전부보다 CV가 높은가?**

💡 **쉽게 말하면**

- '중요한 피처만 남기면 더 좋다'는 **가중치 학습 모델의 사고방식**.
- KNN은 가중치를 학습하지 않고 피처를 전부 거리식에 넣음.

**실행 흐름**

1. 같은 Pipeline · 같은 CV
2. petal 2개 · 전체 4개
3. 평균 · 표준편차 비교

**먼저 예측** · petal 2개가 더 높을까? — 답은 코드 실행 결과 아래에

In [10]:
# 왜 · 특성을 줄여도 점수가 유지되는지 같은 CV로 비교하기 위해
s_all = cross_val_score(knn_pipe(5), X, y, cv=cv, scoring="accuracy")            # 4개
s_petal = cross_val_score(knn_pipe(5), X[:, 2:4], y, cv=cv, scoring="accuracy")  # petal 2개
print(f"petal 2개: {s_petal.mean():.4f} ± {s_petal.std():.4f}")                  # 원본 셀 25
print(f"전체 4개 : {s_all.mean():.4f} ± {s_all.std():.4f}")                      # 원본 셀 25
gap, sd = s_all.mean() - s_petal.mean(), min(s_all.std(), s_petal.std())         # 차이 · 흔들림
print(f"차이 {gap:.4f} vs 작은 쪽 표준편차 {sd:.4f}")                            # 근거 판단

petal 2개: 0.9600 ± 0.0389
전체 4개 : 0.9733 ± 0.0249
차이 0.0133 vs 작은 쪽 표준편차 0.0249


**결과** · 예측 정답은 아님 — 조금 낮지만 표준편차 안

- `petal 2개: 0.9600 ± 0.0389` — 평균이 약간 낮음
- `전체 4개 : 0.9733 ± 0.0249` — 평균이 약간 높음
- `차이 0.0133 vs 작은 쪽 표준편차 0.0249` — 차이가 표준편차보다 작음 → '4개가 확실히 좋다'고 단정 못 함

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>로지스틱</td><td>계수 w를 학습 (파일별 5)</td><td>피처별 중요도를 스스로 판단</td></tr>
<tr><td>트리</td><td>분기 규칙 · feature_importances_ (파일별 6)</td><td>무관한 피처를 안 쓰고 넘어갈 수 있음</td></tr>
<tr><td>KNN</td><td>아무것도 학습 안 함 · 모든 피처를 거리에 합산</td><td>피처를 빼면 단서가 사라질 수도</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 앞에서는 피처가 늘면 '노이즈', 여기서는 줄이면 '실명' → 둘을 가르는 기준은 6-2에서
> - 확인한 것은 '이 데이터에서 sepal을 빼니 평균이 조금 내려갔다' 하나뿐

> ✅ **한 줄 정리** · petal만 남겨도 4개보다 확실히 좋아지지 않았으며, 그 차이는 표준편차 안이라 단정할 근거가 없다.

**딥러닝 코드에서는**

- 입력 피처 제거 실험(ablation)도 seed · fold 흔들림과 함께 읽기

### 4-2. `A` permutation importance · `permutation_importance` · `n_repeats=30` <sub>원본 셀 27 ~ 29</sub>

> 🎯 **계수가 없는 KNN에서 피처를 하나씩 섞으면 성능은 얼마나 떨어지는가?**

💡 **쉽게 말하면**

- 피처 하나를 **무작위로 섞고** 성능이 얼마나 떨어지는지 → 많이 떨어지면 중요.
- 모델 종류를 안 가림 (KNN · 트리 · 딥러닝).

**실행 흐름**

1. 7 : 3 분할 · Pipeline fit
2. test에서 30번씩 섞기
3. 피처별 평균 하락 · 표준편차

**먼저 예측** · sepal 두 개의 중요도도 클까? — 답은 코드 실행 결과 아래에

In [11]:
from sklearn.inspection import permutation_importance  # 섞어서 성능 변화 보기

# 왜 · KNN에는 내장 중요도가 없어 섞어 보기(permutation)로 특성 중요도를 재기 위해
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)  # 원본 셀 28
pipe_pi = knn_pipe(5).fit(Xa, ya)                                    # Scaler + KNN
imp = permutation_importance(pipe_pi, Xb, yb, n_repeats=30,          # test에서 30번씩 섞기
                             random_state=42, scoring="accuracy")
imp_df = pd.DataFrame({"피처": iris.feature_names,                   # 원본 표
                       "중요도(성능 하락폭)": imp.importances_mean, "± std": imp.importances_std})
imp_df = imp_df.sort_values("중요도(성능 하락폭)", ascending=False)  # 큰 순서
print(imp_df.round(4).to_string(index=False))                        # 원본 표

               피처  중요도(성능 하락폭)  ± std
 petal width (cm)       0.2400 0.0442
petal length (cm)       0.2030 0.0576
sepal length (cm)       0.0067 0.0238
 sepal width (cm)      -0.0067 0.0288


**결과** · 예측 정답은 아님 — sepal은 0 근처

- `petal width (cm)               0.2400  0.0442` — 섞으면 정확도가 0.24 떨어짐
- `petal length (cm)               0.2030  0.0576` — 두 번째
- `sepal width (cm)              -0.0067  0.0288` — 0에 붙어 있고 표준편차 안에서 음수도

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>계수 (로지스틱)</td><td>학습된 중요도</td><td>모델 안에 있음</td></tr>
<tr><td>feature_importances_ (트리)</td><td>분기에서 줄인 불순도</td><td>모델 안에 있음</td></tr>
<tr><td>permutation importance</td><td>섞었을 때 성능 하락 · 사후 측정</td><td><strong>모든 모델</strong></td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 'KNN은 피처 중요도를 못 본다'는 틀림 → 학습된 계수가 없을 뿐 사후 측정은 가능
> - 중요도가 0에 가깝다는 것과 빼도 안전하다는 것은 다른 문제

> ✅ **한 줄 정리** · KNN은 학습된 계수가 없지만 permutation importance로 사후에 중요도를 잴 수 있으며, Iris에서는 petal ≫ sepal이었다.

**딥러닝 코드에서는**

- 입력 토큰 · 채널을 섞거나 가려서 기여도를 보는 해석 방법

---
## 5. GridSearchCV

> **이 장의 흐름**
>
> - test를 먼저 봉인하고 dev 120개에서 K 6개 × weights 2개를 5-Fold로
> - 그다음 weights별 train · CV와 공동 1위 수 확인

<img src="images/pfm8_grid.png" alt="weights별 train(점선) vs CV(실선) · distance는 train 1.0" width="620">

**파이프라인 위치** · ⑤ 교차검증 · 튜닝

### 5-1. `A` K × weights 탐색 · distance의 train · `GridSearchCV` · `weights="distance"` <sub>원본 셀 30 ~ 34</sub>

> 🎯 **`distance`로 K = 11을 쓰면 train 정확도는 몇인가? 과적합은 무엇으로 보는가?**

💡 **쉽게 말하면**

- `uniform` = 이웃 K개가 한 표씩 · `distance` = **가까울수록 무거운 표** (거리의 역수).
- 이건 **이웃 투표 가중치** · 피처 가중치는 여전히 학습 안 함.

**실행 흐름**

1. test 20% 먼저 봉인
2. dev 120에서 12조합 × 5-Fold
3. weights별 train · CV 표

**먼저 예측** · distance · K = 11의 train 정확도는? — 답은 코드 실행 결과 아래에

In [12]:
from sklearn.model_selection import GridSearchCV  # 후보 × CV

# 왜 · K와 투표 방식을 dev 안에서만 고르고 최종 test는 마지막에 한 번만 쓰기 위해
X_dev, X_final_test, y_dev, y_final_test = train_test_split(  # 수능 먼저 봉인 (원본 셀 31)
    X, y, test_size=0.2, stratify=y, random_state=42)
param_grid = {"knn__n_neighbors": [1, 3, 5, 7, 9, 11],                            # 6 × 2 = 12조합
              "knn__weights": ["uniform", "distance"]}
gs = GridSearchCV(knn_pipe(), param_grid, scoring="accuracy", cv=cv,              # dev 안에서만
                  return_train_score=True).fit(X_dev, y_dev)
print(f"탐색 범위 (dev) {len(X_dev)}개 · 봉인 (test) {len(X_final_test)}개")      # 120 · 30
print(f"CV 최고 조합 {gs.best_params_} · CV 최고 Accuracy {gs.best_score_:.4f}")  # 순서상 먼저
cv_table = pd.DataFrame(gs.cv_results_).rename(columns={"param_knn__n_neighbors": "K",  # 원본 셀 32
                                                        "param_knn__weights": "weights"})
for w in ["uniform", "distance"]:                                                 # weights별
    sub = cv_table[cv_table["weights"] == w].sort_values("K")                     # K 순
    print(w, "train:", sub["mean_train_score"].round(3).tolist())                 # 외운 문제
    print(w, "CV   :", sub["mean_test_score"].round(3).tolist())                  # 처음 보는 문제

탐색 범위 (dev) 120개 · 봉인 (test) 30개
CV 최고 조합 {'knn__n_neighbors': 3, 'knn__weights': 'uniform'} · CV 최고 Accuracy 0.9667
uniform train: [1.0, 0.969, 0.969, 0.967, 0.954, 0.958]
uniform CV   : [0.942, 0.967, 0.958, 0.933, 0.958, 0.95]
distance train: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
distance CV   : [0.942, 0.967, 0.958, 0.942, 0.958, 0.967]


**결과** · 예측 정답은 1.000 — 자기 자신까지의 거리가 0이라 자기 표가 무한대

- `탐색 범위 (dev) 120개 · 봉인 (test) 30개` — 수능은 맨 마지막에 한 번
- `CV 최고 조합 {'knn__n_neighbors': 3, 'knn__weights': 'uniform'} · CV 최고 Accuracy 0.9667` — 순서상 먼저인 후보
- `distance train: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]` — K와 상관없이 train 1.000
- `uniform train: [1.0, 0.969, 0.969, 0.967, 0.954, 0.958]` — K가 커지면 train이 내려감

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>uniform</code></td><td>이웃 K개가 한 표씩</td><td>기본 · 더 단순</td></tr>
<tr><td><code>distance</code></td><td>가까울수록 표가 무거움</td><td>이웃 거리 차이가 클 때</td></tr>
<tr><td>과적합 판단</td><td>train 점수가 아니라 <strong>train-CV 격차</strong></td><td>distance는 train이 늘 1.0</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 'K를 키우면 과적합이 줄어든다'는 `uniform`일 때의 이야기 → distance에서는 train이 항상 1.0
> - 슬라이드의 'KNN은 가중치가 없다'는 **피처 가중치** 이야기 · `weights=`는 **이웃 투표**

> ✅ **한 줄 정리** · `weights='distance'`는 train에서 자기 자신의 표가 무한대라 train이 항상 1.0이므로, 과적합은 train-CV 격차로 판단한다.

**딥러닝 코드에서는**

- attention의 가중 평균도 '가까운(비슷한) 것에 더 큰 표'라는 같은 생각

### 5-2. `A` 공동 1위 · `rank_test_score == 1` <sub>원본 셀 34 ~ 36</sub>

> 🎯 **rank_test_score = 1인 조합은 몇 개이고, 그중 무엇을 골라야 하는가?**

💡 **쉽게 말하면**

- `best_params_`는 하나만 돌려주지만 실제로는 **공동 1위가 여럿**.
- 동점이면 먼저 계산된 것을 고름 → 성능 우열이 아님.

**실행 흐름**

1. rank 1인 행
2. K · weights · 평균 · 표준편차

**먼저 예측** · 공동 1위는 몇 개? — 답은 코드 실행 결과 아래에

In [13]:
# 왜 · 점수가 같은 후보가 여럿일 때 어떤 규칙으로 골랐는지 확인하기 위해
cols = ["K", "weights", "mean_test_score", "std_test_score"]  # 볼 열
ties = cv_table[cv_table["rank_test_score"] == 1][cols]       # rank 1인 행
print(f"공동 1위 조합 수: {len(ties)}개")                     # 원본 셀 35
print(ties.round(4).to_string(index=False))                   # 누구인지
print("best_params_ 가 고른 것:", gs.best_params_)            # 동점 중 순서상 먼저

공동 1위 조합 수: 3개
 K  weights  mean_test_score  std_test_score
 3  uniform           0.9667          0.0167
 3 distance           0.9667          0.0167
11 distance           0.9667          0.0167
best_params_ 가 고른 것: {'knn__n_neighbors': 3, 'knn__weights': 'uniform'}


**결과** · 예측 정답은 3개

- `공동 1위 조합 수: 3개` — 승자가 여럿
- `3 uniform           0.9667          0.0167` — best_params_가 고른 것
- `11 distance           0.9667          0.0167` — 같은 점수 · 같은 표준편차

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>더 단순한 쪽</td><td><code>uniform</code>이 <code>distance</code>보다 단순</td><td>기본 선택</td></tr>
<tr><td>K가 큰 쪽</td><td>노이즈에 덜 민감</td><td>안정성을 원할 때</td></tr>
<tr><td>다시 재기</td><td><code>RepeatedStratifiedKFold</code>로 반복 CV</td><td>차이가 진짜인지 확인</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 원본 셀 34는 'K=5의 두 weights가 공동 1위'라고 적었지만 실제 출력의 공동 1위는 K=3 uniform · K=3 distance · K=11 distance
> - 하면 안 되는 말 '최적 조합은 K=3, uniform' · 해야 하는 말 '현재 CV에서는 세 조합이 공동 1위'

> ✅ **한 줄 정리** · GridSearchCV의 best_params_는 동점 후보 중 먼저 나온 하나일 뿐이며, 공동 1위가 여럿이면 더 단순한 쪽을 고르거나 반복 CV로 다시 잰다.

**딥러닝 코드에서는**

- 하이퍼파라미터 탐색 결과가 동점이면 더 작은 모델 · 더 단순한 설정을 고름

---
## 6. 차원의 저주

> **이 장의 흐름**
>
> - 난수 데이터로 차원별 '가장 가까운 거리 ÷ 가장 먼 거리' → 분포를 바꿔 다시
> - 그다음 Iris에 정답과 무관한 난수 컬럼을 붙여 성능 변화

<img src="images/pfm8_noise.png" alt="무관한 컬럼을 붙일수록 무너지는 KNN CV" width="587">

**파이프라인 위치** · ③ 피처 · 차원 — 거리의 의미

### 6-1. `B` 가까운 거리 ÷ 먼 거리 · `cdist` · `make_blobs` <sub>원본 셀 37 · 38 · 42</sub>

> 🎯 **차원이 커지면 가장 가까운 거리 ÷ 가장 먼 거리는 1에 가까워지는가? 분포에 따라 다른가?**

💡 **쉽게 말하면**

- 차원이 늘면 모든 점이 **비슷한 거리**로 퍼짐 → 가까운 것도 멀고, 먼 것도 멂.
- 단, **구조(군집)가 있으면** 거의 안 무너짐.

**실행 흐름**

1. 균등 · 정규 · 군집 데이터 200개
2. 차원 2 · 10 · 50 · 200
3. 점마다 min ÷ max 평균

**먼저 예측** · 균등 난수 200차원의 비율은 1일까? — 답은 코드 실행 결과 아래에

In [14]:
from scipy.spatial.distance import cdist  # 모든 쌍의 거리
from sklearn.datasets import make_blobs   # 군집이 뚜렷한 데이터


# 왜 · 차원이 커질수록 가장 가까운 점과 먼 점의 거리가 비슷해지는 현상을 숫자로 보기 위해
def min_max_ratio(Xr):                  # 점마다 (가장 가까운 거리 ÷ 가장 먼 거리)의 평균
    D = cdist(Xr, Xr)                   # 200 × 200 거리
    np.fill_diagonal(D, np.inf)         # 자기 자신 제외
    mn = D.min(axis=1)                  # 가장 가까운 거리
    np.fill_diagonal(D, 0)              # 최댓값 계산용
    return (mn / D.max(axis=1)).mean()  # 비율 평균


# 왜 · 데이터 분포에 따라 차원의 저주가 다르게 나타나는지 비교하기 위해
rng_cd = np.random.RandomState(42)                                      # 원본 셀 38
for name in ["균등난수", "정규분포", "군집 뚜렷"]:                      # 분포 3가지
    ratios = []                                                         # 차원마다
    for d in [2, 10, 50, 200]:                                          # 차원
        if name == "균등난수":                                          # 원본 셀 38
            Xr = rng_cd.rand(200, d)                                    # 0 ~ 1 균등
        elif name == "정규분포":                                        # 원본 셀 42 표
            Xr = rng_cd.randn(200, d)                                   # 표준정규
        else:                                                           # 구조가 있는 데이터
            Xr, _ = make_blobs(n_samples=200, n_features=d, centers=5,  # 5개 군집
                               cluster_std=0.5, random_state=42)
        ratios.append(round(float(min_max_ratio(Xr)), 3))               # 비율
    print(f"{name}: d=2 · 10 · 50 · 200 → {ratios}")                    # 한 줄

균등난수: d=2 · 10 · 50 · 200 → [0.034, 0.347, 0.645, 0.805]
정규분포: d=2 · 10 · 50 · 200 → [0.037, 0.322, 0.614, 0.799]
군집 뚜렷: d=2 · 10 · 50 · 200 → [0.011, 0.039, 0.063, 0.073]


**결과** · 예측 정답은 아님 — 0.805 · 군집이 있으면 0.073

- `균등난수: d=2 · 10 · 50 · 200 → [0.034, 0.347, 0.645, 0.805]` — 차원이 커질수록 1에 다가가지만 200차원에서도 0.805
- `정규분포: d=2 · 10 · 50 · 200 → [0.037, 0.322, 0.614, 0.799]` — 균등과 비슷하게 커짐
- `군집 뚜렷: d=2 · 10 · 50 · 200 → [0.011, 0.039, 0.063, 0.073]` — 구조가 있으면 200차원에서도 0.073

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>구조 없는 데이터 (난수)</td><td>차원이 늘면 거리 구별력 붕괴</td><td>차원의 저주가 심함</td></tr>
<tr><td>구조 있는 데이터 (군집)</td><td>차원이 늘어도 가까운 것은 가까움</td><td>덜 무너짐</td></tr>
<tr><td>min ÷ max → 1</td><td>직관을 위한 그림</td><td>보편 법칙이 아님</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 원본 셀 42의 분포별 표(정규 · 군집)는 생성 코드가 노트북에 없어, 여기서는 정규 = randn · 군집 = make_blobs(5개 군집)로 다시 만듦 → 숫자는 원본과 다를 수 있음
> - 원본 표의 값(균등 0.036 · 0.332 · 0.644 · 0.807 · 군집 200차원 0.103)과 끝자리는 달라도 같은 방향

> ✅ **한 줄 정리** · 차원이 늘면 구조 없는 데이터에서 가까운 거리와 먼 거리의 차이가 줄어들지만, 구조가 있으면 거의 무너지지 않는다.

**딥러닝 코드에서는**

- 고차원 임베딩이 쓸 만한 이유 = 학습으로 의미 있는 구조(군집)를 만들기 때문

### 6-2. `A` 무관한 컬럼을 붙이면 · `np.hstack` · `noise_pool` <sub>원본 셀 39 ~ 42</sub>

> 🎯 **Iris에 정답과 무관한 난수 컬럼을 2 · 20 · 200개 붙이면 KNN은 알아서 무시하는가?**

💡 **쉽게 말하면**

- KNN은 가중치를 학습하지 않음 → **쓸모없는 컬럼도 거리에 그대로 합산**.
- 진짜 신호(petal)가 노이즈 합계에 파묻힘.

**실행 흐름**

1. seed 5개 × 노이즈 수 8가지
2. Pipeline + 5-Fold
3. 평균 · 표준편차

**먼저 예측** · 노이즈 200개면 랜덤 추측(0.333)까지 내려갈까? — 답은 코드 실행 결과 아래에

In [15]:
# 왜 · 쓸모없는 특성이 늘수록 KNN 점수가 떨어지는 것을 확인하기 위해
n_list = [0, 2, 5, 10, 20, 50, 100, 200]                               # 붙일 노이즈 수 (원본 셀 40)
rows = []                                                              # (seed, 노이즈 수, 정확도)
for seed in range(5):                                                  # seed 5개
    noise_pool = np.random.RandomState(seed).rand(len(X), max(n_list))  # 노이즈 200개를 한 번만
    cv_noise = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)  # seed마다 CV
    for n in n_list:                                                   # 앞에서부터 잘라 씀 (누적)
        X_aug = np.hstack([X, noise_pool[:, :n]])                      # 원래 4개 + 노이즈 n개
        rows.append([seed, n, cross_val_score(knn_pipe(5), X_aug, y, cv=cv_noise).mean()])  # CV
noise_df = pd.DataFrame(rows, columns=["seed", "노이즈 수", "acc"])    # 40행
noise_df = noise_df.groupby("노이즈 수")["acc"]                        # 노이즈 수별
print(noise_df.agg(평균="mean", 표준편차="std").round(4).to_string())  # 원본 표

           평균    표준편차
노이즈 수                
0      0.9520  0.0073
2      0.8933  0.0200
5      0.8360  0.0252
10     0.8053  0.0242
20     0.7507  0.0361
50     0.6867  0.0641
100    0.6027  0.0289
200    0.5253  0.0292


**결과** · 예측 정답은 0.333까지는 아니지만 0.53으로 붕괴

- `0  0.9520  0.0073` — 노이즈 없음
- `50  0.6867  0.0641` — +50개 → 크게 하락
- `200  0.5253  0.0292` — +200개 → 붕괴 (오차막대 밖)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>sepal 2개 빼기 (4-1)</td><td>0.9733 → 0.9600 · std 안</td><td>판단 보류</td></tr>
<tr><td>노이즈 200개 붙이기</td><td>0.9520 → 0.5253 · std 밖</td><td><strong>명백한 노이즈</strong></td></tr>
<tr><td>기준</td><td>피처 개수가 아니라 정답과 관계가 있는가</td><td>오차막대까지 보고 판단</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 차원이 늘어서 무너진 게 아니라 **정답과 무관한 차원**이 늘어서 무너진 것
> - 어떤 피처가 무관한지는 데이터마다 확인해야 앎 · 고차원이면 차원 축소 · 피처 선택 · 다른 모델

> ✅ **한 줄 정리** · 정답과 무관한 컬럼은 KNN의 거리에 그대로 더해져 진짜 신호를 파묻으므로, 붙일수록 성능이 무너진다.

**딥러닝 코드에서는**

- 입력이 고차원이면 먼저 표현을 학습(임베딩)한 뒤 그 공간에서 이웃을 찾음

---
## 7. 종합 실습 · 수능 보러 가기 · 펭귄 0.7801 다시 설명

> **이 장의 흐름**
>
> - 5장에서 고른 모델로 Iris test 30개를 한 번만 평가
> - 그다음 펭귄 342마리에 스케일링 + K 탐색을 적용해 0.7801을 다시 설명

<img src="images/pfm8_practice.png" alt="펭귄 KNN · 스케일링 전후 · K별 CV" width="575">

**파이프라인 위치** · ⑤ 선택 완료 → ⑥ test 1회 · 다른 데이터로 확장

> 🎯 **Iris test 점수는 CV보다 높을까 낮을까? 펭귄 KNN은 스케일링과 K 선택으로 얼마나 올라가는가?**

💡 **쉽게 말하면**

- test 30개 → 1명 틀리면 0.0333씩 움직임.
- 펭귄 0.78의 원인은 **단위**(체중 g) · K는 그 다음.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>Iris · dev에서 고른 모델로 test 1회</td><td>5-1</td></tr>
<tr><td>2</td><td>펭귄 측정값 4개 · 같은 분할(seed 7)</td><td>파일별 2</td></tr>
<tr><td>3</td><td>스케일링 없음 vs Pipeline · K 1 ~ 15 CV</td><td>2 · 3장</td></tr>
<tr><td>4</td><td>고른 K로 펭귄 test 1회</td><td>7장 1단계</td></tr>
</tbody></table>

**먼저 예측** · 펭귄 KNN CV는 스케일링 + 좋은 K로 몇까지 오를까? — 답은 코드 실행 결과 아래에

In [16]:
# 1단계: Iris test 1회 (dev에서 이미 고름)
# 왜 · dev에서 고른 설정 그대로 최종 test를 한 번만 채점하기 위해
final_test_acc = gs.best_estimator_.score(X_final_test, y_final_test)        # 처음이자 마지막
print(f"CV 최고 (dev 안) : {gs.best_score_:.4f}")                            # 고를 때 쓴 점수
print(f"test (수능)      : {final_test_acc:.4f}   (n={len(y_final_test)})")  # 보고
print(f"test에서 1명 틀릴 때 정확도 변화폭: {1 / len(y_final_test):.4f}")    # 해상도

CV 최고 (dev 안) : 0.9667
test (수능)      : 0.9333   (n=30)
test에서 1명 틀릴 때 정확도 변화폭: 0.0333


**실행 흐름**

1. 파일별 2와 같은 펭귄 데이터 · 분할 준비
2. 스케일링 없는 K=5 CV(0.7801) 다시 계산
3. 스케일링 + K별 CV로 K 고르기
4. 고른 K로 펭귄 test 1회

In [17]:
# 2 · 3단계: 펭귄 KNN 0.7801 다시 설명 (파일별 2와 같은 데이터 · 분할)
# 왜 · 앞 권의 낮은 KNN 점수가 스케일링 부족 때문이었는지 같은 조건으로 확인하기 위해
meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]  # 측정값 4개
peng = pd.read_csv("../02_모델선택_펭귄/penguins.csv").dropna(subset=meas)  # 342마리
Xp, yp = peng[meas].to_numpy(), peng["species"].to_numpy()          # 특성 · 품종
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(                      # 파일별 2와 같은 분할
    Xp, yp, test_size=0.2, stratify=yp, random_state=7)
# 0.7801 재현
print(f"스케일링 없음 K=5 CV: {cross_val_score(KNN(5), Xp_tr, yp_tr, cv=5).mean():.4f}")
k_cv = {k: cross_val_score(knn_pipe(k), Xp_tr, yp_tr, cv=5).mean()  # 스케일링 + K마다 CV
        for k in [1, 3, 5, 7, 9, 11, 15]}
print("스케일링 + K별 CV:", {k: round(float(v), 4) for k, v in k_cv.items()})  # K 비교
best_k = max(k_cv, key=k_cv.get)                                    # 가장 높은 K (동점이면 먼저)


# 4단계: 고른 K로 펭귄 test 1회
# 왜 · 고른 K 그대로 test를 한 번만 써서 공정하게 비교하기 위해
peng_test = knn_pipe(best_k).fit(Xp_tr, yp_tr).score(Xp_te, yp_te)        # 처음이자 마지막
print(f"고른 K={best_k} · CV {k_cv[best_k]:.4f} · test {peng_test:.4f}")  # 보고

스케일링 없음 K=5 CV: 0.7801
스케일링 + K별 CV: {1: 0.9818, 3: 0.9818, 5: 0.9854, 7: 0.9854, 9: 0.9818, 11: 0.9818, 15: 0.9818}
고른 K=5 · CV 0.9854 · test 1.0000


**결과** · 예측 정답은 Iris test 0.9333 · 펭귄 CV 0.7801 → 0.9854

- `CV 최고 (dev 안) : 0.9667` — 고를 때 쓴 점수
- `test (수능)      : 0.9333   (n=30)` — 1명 차이(0.0333) 수준 · CV보다 낮음
- `스케일링 없음 K=5 CV: 0.7801` — 파일별 2 · 3과 같은 값 (출발점)
- `스케일링 + K별 CV: {1: 0.9818, 3: 0.9818, 5: 0.9854, 7: 0.9854, 9: 0.9818, 11: 0.9818, 15: 0.9818}` — 스케일링만으로 0.98대 · K 차이(0.004)는 그보다 훨씬 작음
- `고른 K=5 · CV 0.9854 · test 1.0000` — K=5 · 7 동점 중 먼저 · 펭귄 test 69마리 전부 정답 · 다시 고르지 않음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>4-1 로지스틱</td><td>valid로 임곗값 · 피처</td><td>test 1회</td></tr>
<tr><td>4-2 KNN (이 노트북)</td><td>CV로 스케일링 · 피처 · K · weights</td><td>test 1회</td></tr>
<tr><td>공통 원칙</td><td>고르는 데 쓴 점수는 성능이 아님</td><td>모델이 확률이든 거리든 같음</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 원본 셀 46은 '0.9667과 1.0000의 차이는 한 명'이라 적었지만 이 test 점수는 0.9333 (원본 출력도 0.9333)
> - test 30개라 이 숫자 하나로는 확정 못 함 · '100%면 완벽'도 같은 이유로 하면 안 되는 말

> ✅ **한 줄 정리** · KNN을 잘 쓴다는 것은 모델을 튜닝하는 것보다 좋은 이웃을 찾을 수 있는 거리(단위 · 피처 · 차원)를 설계하는 일이다.

**딥러닝 코드에서는**

- 임베딩 · 정규화 · 차원 축소로 '좋은 거리 공간'을 먼저 만들고 최근접 이웃을 찾음

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. KNN 원리</td><td>거리 직접 계산 = sklearn 이웃 · test 1.000 · 3 × 3 혼동행렬</td><td>1-1 ~ 1-3</td></tr>
<tr><td>2. 스케일링</td><td>0.650 → 0.983 · Iris 한 번 split 1.000 → 0.933은 착시 (5-Fold 0.9667 &lt; 0.9733)</td><td>2-1 ~ 2-3</td></tr>
<tr><td>3. K · 경계</td><td>K=1 train 1.0 · 격차 0.0583 · K=3 CV 최고</td><td>3-1 · 3-2</td></tr>
<tr><td>4. weights · 공동 1위</td><td>distance는 train 항상 1.0 · 공동 1위 3개</td><td>5-1 · 5-2</td></tr>
<tr><td>5. 차원의 저주</td><td>노이즈 200개 → 0.9520 → 0.5253 · 펭귄 0.7801 → 스케일링으로 0.9854</td><td>6-2 · 7</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>KNN</td><td>가까운 K개 다수결 · 학습 = 저장</td><td>최근접 이웃 검색</td></tr>
<tr><td><code>A</code></td><td>유클리드 거리</td><td>√(Σ 차이²) · 큰 단위가 지배</td><td><code>torch.cdist</code></td></tr>
<tr><td><code>A</code></td><td>스케일링 · Pipeline</td><td>거리 기준을 맞춤 · train에만 fit</td><td>입력 정규화</td></tr>
<tr><td><code>A</code></td><td>K</td><td>작으면 과적합 · 크면 과소적합 · CV로</td><td>모델 크기</td></tr>
<tr><td><code>A</code></td><td>train-CV 격차</td><td>과적합 판단 기준</td><td>train · val 곡선</td></tr>
<tr><td><code>A</code></td><td>차원의 저주</td><td>무관한 차원이 거리를 파묻음</td><td>임베딩으로 구조 만들기</td></tr>
<tr><td><code>B</code></td><td>weights</td><td>uniform 한 표 · distance 가까울수록 무거움</td><td>attention 가중 평균</td></tr>
<tr><td><code>B</code></td><td>공동 1위</td><td>best_params_는 동점 중 하나</td><td>단순한 설정 선택</td></tr>
<tr><td><code>B</code></td><td>permutation importance</td><td>계수 없이도 사후 측정</td><td>해석 도구</td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · KNN은 학습 때 무엇을 하는가?
- **1-2** · 3 × 3 혼동행렬의 대각선은?
- **1-3** · 손으로 찾은 이웃과 sklearn 이웃은?
- **2-1** · 0 ~ 1 vs 0 ~ 1000 데이터의 스케일링 효과는?
- **2-2** · Iris 한 번 split에서 스케일링 후 정확도는?
- **2-3** · 그 하락은 진짜인가?
- **3-1** · K = 1의 train 정확도는?
- **3-2** · K가 작으면 경계는?
- **4-1** · petal 2개 vs 4개 차이는 믿을 만한가?
- **4-2** · KNN도 피처 중요도를 볼 수 있나?
- **5-1** · distance의 train이 항상 1.0인 이유는?
- **5-2** · 공동 1위는 몇 개?
- **6-1** · min ÷ max 비율은 늘 1로 가나?
- **6-2** · 무관한 컬럼 200개를 붙이면?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. Iris 전체에 `knn_pipe(7, "distance")`의 5-Fold CV 평균을 소수 넷째 자리로 출력
2. 같은 설정에서 petal 2개만 쓴 CV 평균을 소수 넷째 자리로 출력

In [18]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 2-1에서 feature2 범위를 0 ~ 1000 → 0 ~ 10으로 바꾸면 스케일링 없음의 정확도는?
2. 6-2에서 노이즈를 0 ~ 1 균등 대신 정답과 관련된 열(petal length + 작은 노이즈)로 붙이면?
3. 7장 펭귄에서 `MinMaxScaler`를 쓰면 K=5 CV는?

In [19]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · 데이터를 저장 · 예측 때 가까운 K개 다수결
- **1-2** · 맞힌 수 · 나머지는 헷갈린 쌍
- **1-3** · 같음 (거리 계산 + 다수결)
- **2-1** · 0.650 → 0.983
- **2-2** · 1.000 → 0.933
- **2-3** · 아님 · 20번 중 11번 동점 · 5-Fold는 0.9667 &lt; 0.9733
- **3-1** · 1.000 (자기 자신이 이웃)
- **3-2** · 울퉁불퉁 · 과적합
- **4-1** · 아님 · 차이 0.0133 &lt; 표준편차
- **4-2** · permutation importance로 사후 측정 · petal ≫ sepal
- **5-1** · 자기 자신까지의 거리 0 → 자기 표가 무한대
- **5-2** · 3개 · 단순한 쪽을 고르거나 반복 CV
- **6-1** · 아님 · 구조가 있으면 작게 유지
- **6-2** · 0.9520 → 0.5253으로 붕괴

#### 코드 문제 답

```python
print(round(cross_val_score(knn_pipe(7, "distance"), X, y, cv=cv).mean(), 4))  # 0.9667
print(round(cross_val_score(knn_pipe(7, "distance"), X[:, 2:4], y, cv=cv).mean(), 4))  # 0.9533
```

#### 변형 문제 답

*문제 → 답*

- **1** · 0.9667 — 범위 차이가 10배로 줄어 feature1도 거리에 반영됨 (0 ~ 1000일 때 0.650)
- **2** · petal length + 작은 노이즈 열 20개를 붙여도 0.96 (안 붙임 0.9533 · 같은 CV seed 0) — 정답과 관계있는 차원은 노이즈가 아님
- **3** · 0.9854 — Standard와 같음 · 스케일링 여부가 스케일러 종류보다 중요 (파일별 3)

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) 한 번의 split에서 스케일링 후 하락 → 스케일링은 안 하는 게 낫다고 답함</td><td>한 번의 결과로 결론</td><td>여러 split · CV로 다시 재기</td><td>seed를 바꿔도 같은 결과인가?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. KNN에서 스케일링이 왜 필수인가요?**
> - KNN은 거리로 이웃을 찾는데, 단위가 큰 특성이 거리를 지배하면 다른 특성이 무시됩니다.
> - 범위가 0에서 1과 0에서 1000인 데이터에서 스케일링 전 0.650이 스케일링 후 0.983이 됐고, 펭귄도 체중 때문에 0.78이던 CV가 0.98대로 올랐습니다.
> - 스케일러는 Pipeline에 넣어 train에만 fit합니다.

**Q2. K는 어떻게 고르나요?**
> - K가 작으면 경계가 점마다 휘어 과적합되고, 너무 크면 세부 패턴을 놓칩니다.
> - train과 교차검증 정확도의 격차를 보며 GridSearchCV로 고릅니다.
> - 공동 1위가 여럿이면 best_params_는 그중 하나일 뿐이라, 더 단순한 설정을 고르거나 반복 CV로 다시 잽니다.
> - weights가 distance면 train이 항상 1.0이라 격차로 판단합니다.

**Q3. 차원의 저주란 무엇이고 KNN에 왜 치명적인가요?**
> - 차원이 늘면 구조 없는 데이터에서 가장 가까운 점과 가장 먼 점의 거리 차이가 줄어 이웃의 의미가 약해집니다.
> - KNN은 피처 가중치를 학습하지 않아 무관한 컬럼도 거리에 그대로 더하므로, Iris에 난수 컬럼 200개를 붙이자 0.95가 0.53으로 무너졌습니다.
> - 그래서 차원 축소나 피처 선택, 다른 모델을 고려합니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>KNeighborsClassifier(n_neighbors=, weights=, p=)</code></td><td>sklearn.neighbors</td><td>KNN</td><td>1-1 <code>KNN(n_neighbors=5, metric="minkowski", p=2)</code></td></tr>
<tr><td><code>kneighbors</code></td><td>KNN 메서드</td><td>이웃 번호 · 거리</td><td>1-3 <code>model.kneighbors([star])</code></td></tr>
<tr><td><code>np.linalg.norm(axis=1)</code> · <code>np.argsort</code></td><td>numpy</td><td>거리 · 가까운 순</td><td>1-3 <code>np.linalg.norm(X_train - star, axis=1)</code></td></tr>
<tr><td><code>confusion_matrix</code> · <code>np.trace</code></td><td>sklearn.metrics · numpy</td><td>3 × 3 표 · 대각선 합</td><td>1-2 <code>np.trace(cm)</code></td></tr>
<tr><td><code>StandardScaler</code> · <code>Pipeline</code></td><td>sklearn</td><td>스케일링 · 누수 방지</td><td>2-1 <code>StandardScaler().fit(Xd_tr)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>cross_val_score</code></td><td>sklearn.model_selection</td><td>교차검증</td><td>2-3 <code>cross_val_score(knn_pipe(5), X, y, cv=cv)</code></td></tr>
<tr><td><code>permutation_importance</code></td><td>sklearn.inspection</td><td>섞었을 때 성능 하락</td><td>4-2 <code>permutation_importance(pipe_pi, Xb, yb, …)</code></td></tr>
<tr><td><code>GridSearchCV(return_train_score=True)</code></td><td>sklearn.model_selection</td><td>K × weights 탐색</td><td>5-1 <code>GridSearchCV(knn_pipe(), param_grid, …)</code></td></tr>
<tr><td><code>rank_test_score</code></td><td>GridSearchCV 결과</td><td>공동 1위 확인</td><td>5-2 <code>cv_table["rank_test_score"] == 1</code></td></tr>
<tr><td><code>cdist</code></td><td>scipy.spatial.distance</td><td>모든 쌍의 거리</td><td>6-1 <code>D = cdist(Xr, Xr)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>np.hstack</code></td><td>numpy</td><td>노이즈 컬럼 붙이기</td><td>6-2 <code>np.hstack([X, noise_pool[:, :n]])</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>KNN 앞에 스케일러를 Pipeline으로</td><td>단위가 다른 특성을 그대로</td></tr>
<tr><td>한 번의 split 결과는 CV로 다시</td><td>한 번의 하락으로 결론</td></tr>
<tr><td>과적합은 train-CV 격차로</td><td>distance의 train 1.0을 과적합으로 오해</td></tr>
<tr><td>공동 1위면 단순한 쪽</td><td>best_params_를 유일한 승자로</td></tr>
<tr><td>무관한 컬럼은 확인 후 제거</td><td>피처를 많이 넣으면 무조건 좋다고 가정</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td>로지스틱 vs KNN</td><td>확률 · 가중치 학습 vs 거리 · 학습 없음</td><td>KNN은 스케일링 필수</td></tr>
<tr><td>uniform vs distance</td><td>한 표씩 vs 가까울수록 무거움</td><td>distance는 train 1.0</td></tr>
<tr><td>피처 가중치 vs 이웃 가중치</td><td>KNN은 학습 안 함 vs <code>weights=</code>로 직접 정함</td><td>서로 다른 것</td></tr>
<tr><td>실명 vs 노이즈 제거</td><td>유용한 피처 삭제 vs 무관한 피처 삭제</td><td>오차막대로 판단</td></tr>
<tr><td>CV 점수 vs test 점수</td><td>고를 때 쓴 점수 vs 보고할 점수</td><td>보고는 test 1회</td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>15분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 · 1-3 거리 계산을 다른 꽃으로</td><td>20분</td></tr>
<tr><td>7일 뒤</td><td>7장 펭귄 실습을 빈 셀에 처음부터 · weights까지 GridSearch로</td><td>30분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 보강 2(건포도 과제) 시작 전 ✅ 줄만 읽기</td><td>15분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 노트북의 Part 1 ~ 8을 1 ~ 6장 · 14개 섹션 · 7장 종합 실습으로 나눔
- 원본 plotly 그림 6개(혼동행렬 · K 곡선 · 경계 드롭다운 · GridSearch 곡선 · 거리 비율 · 노이즈 곡선)는 장 그림과 결과 줄로
- 원본 셀 42의 분포별 표(생성 코드 없음)는 6-1에서 코드로 다시 만듦 · 슬라이드 스크린샷 9장은 💡 · 비교 표의 글로
- 7장에 원본이 질문으로 남긴 '펭귄 0.7801 다시 설명'을 실제로 풀어 봄 (새로 추가)

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>셀 0 ~ 2</td><td>목표 · 로지스틱 vs KNN · 척추 질문</td><td>0. 원본 목표 · 힌트</td></tr>
<tr><td>셀 3 ~ 6</td><td>KNN 원리 · 거리 · Iris 기본</td><td>1-1 · 1-3</td></tr>
<tr><td>셀 7 ~ 9</td><td>혼동행렬</td><td>1-2</td></tr>
<tr><td>셀 10 ~ 12</td><td>스케일링의 중요성 · 0 ~ 1 vs 0 ~ 1000</td><td>2-1 · 2장 그림</td></tr>
<tr><td>셀 12 · 13</td><td>Pipeline · 한 번 split</td><td>2-2</td></tr>
<tr><td>셀 14 ~ 17</td><td>split 20번 · CV · 판단 기준 Q&amp;A</td><td>2-3</td></tr>
<tr><td>셀 18 ~ 20</td><td>K별 train vs CV</td><td>3-1</td></tr>
<tr><td>셀 21 ~ 23</td><td>Decision Boundary</td><td>3-2 · 3장 그림</td></tr>
<tr><td>셀 24 ~ 26</td><td>petal 2개 vs 4개 · 실명 vs 노이즈</td><td>4-1</td></tr>
<tr><td>셀 27 ~ 29</td><td>permutation importance</td><td>4-2 · 4장 그림</td></tr>
<tr><td>셀 30 ~ 34</td><td>GridSearchCV · weights · distance의 train</td><td>5-1 · 5장 그림</td></tr>
<tr><td>셀 34 ~ 36</td><td>공동 1위</td><td>5-2</td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>셀 37 · 38 · 42</td><td>차원의 저주 · 거리 비율 · 분포별 표</td><td>6-1</td></tr>
<tr><td>셀 39 ~ 42</td><td>노이즈 컬럼 실험 · 질문의 답</td><td>6-2 · 6장 그림</td></tr>
<tr><td>셀 44 ~ 46</td><td>수능 · 4-1과 4-2가 같은 말</td><td>7. 종합 실습 1단계</td></tr>
<tr><td>셀 41 (질문)</td><td>펭귄 0.7801 다시 설명</td><td>7. 종합 실습 2 ~ 4단계</td></tr>
<tr><td>셀 47 · 48</td><td>오늘의 정리 · 최종 비교 · 다음 예고</td><td>그래서 오늘 · 최종 요약표</td></tr>
</tbody></table>

### 전체 코드

1 ~ 7장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 기본 KNN
2. 1-2 혼동행렬
3. 1-3 이웃 5개 직접 계산

In [20]:
# ---------- 1-1 ----------
# 왜 · 기본 KNN의 학습 · 예측 · 채점 흐름을 먼저 확인하기 위해
X_train, X_test, y_train, y_test = train_test_split(            # 원본 셀 6
    X, y, test_size=0.2, stratify=y, random_state=42)
model = KNN(n_neighbors=5, metric="minkowski", p=2)             # p=2 → 유클리드 거리
model.fit(X_train, y_train)                                     # 학습 = 데이터 저장
y_pred = model.predict(X_test)                                  # 이웃 5개 다수결
print(f"[Test Accuracy] {accuracy_score(y_test, y_pred):.3f}")  # 원본 형식
print(classification_report(y_test, y_pred, target_names=iris.target_names, digits=3))  # 클래스별


# ---------- 1-2 ----------
# 왜 · 어느 품종끼리 헷갈리는지 혼동행렬로 확인하기 위해
cm = confusion_matrix(y_test, y_pred)                      # 행 = 실제, 열 = 예측
print(pd.DataFrame(cm, index=iris.target_names, columns=iris.target_names).to_string())  # 표
print("대각선 합(맞힌 수):", np.trace(cm), "/", cm.sum())  # 30 / 30


# ---------- 1-3 ----------
# 왜 · KNN이 가장 가까운 이웃 5개의 다수결로 예측한다는 것을 직접 계산해 보기 위해
star = X_test[0]                                    # 새 꽃 한 송이
dists = np.linalg.norm(X_train - star, axis=1)      # train 120송이까지 거리
near = np.argsort(dists)[:5]                        # 가장 가까운 5개 번호
print("새 꽃 측정값:", star.tolist(), "· 실제 품종:", iris.target_names[y_test[0]])  # 새 꽃
print("이웃 거리:", dists[near].round(3).tolist())  # 가까운 순
print("이웃 라벨:", y_train[near].tolist(), "→ 다수결", np.bincount(y_train[near]).argmax())  # 투표
_, sk_idx = model.kneighbors([star])                # sklearn이 찾은 이웃
print("sklearn 이웃과 같음:", sorted(sk_idx[0].tolist()) == sorted(near.tolist()))  # 일치 확인

[Test Accuracy] 1.000
              precision    recall  f1-score   support

      setosa      1.000     1.000     1.000        10
  versicolor      1.000     1.000     1.000        10
   virginica      1.000     1.000     1.000        10

    accuracy                          1.000        30
   macro avg      1.000     1.000     1.000        30
weighted avg      1.000     1.000     1.000        30

            setosa  versicolor  virginica
setosa          10           0          0
versicolor       0          10          0
virginica        0           0         10
대각선 합(맞힌 수): 30 / 30
새 꽃 측정값: [4.4, 3.0, 1.3, 0.2] · 실제 품종: setosa
이웃 거리: [0.141, 0.245, 0.3, 0.3, 0.361]
이웃 라벨: [0, 0, 0, 0, 0] → 다수결 0
sklearn 이웃과 같음: True


**실행 흐름**

1. 2-1 단위 문제 재현
2. 2-2 · 2-3 스케일링 효과 (분할 20번)

In [21]:
# ---------- 2-1 ----------
# 왜 · 단위가 큰 특성이 거리를 지배하는 문제를 가짜 데이터로 재현하기 위해
rng_scale = np.random.RandomState(42)                                        # 원본 셀 11
X_demo = np.c_[rng_scale.rand(200), rng_scale.rand(200) * 1000]              # 0 ~ 1 · 0 ~ 1000
y_demo = (X_demo[:, 0] + X_demo[:, 1] / 1000 > 1.0).astype(int)  # 두 특성을 같은 비중으로 쓰는 규칙
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(                               # 7 : 3
    X_demo, y_demo, test_size=0.3, stratify=y_demo, random_state=42)
acc_raw = accuracy_score(yd_te, KNN(5).fit(Xd_tr, yd_tr).predict(Xd_te))     # 그대로
sc = StandardScaler().fit(Xd_tr)                                             # train에만 fit
Xs_tr, Xs_te = sc.transform(Xd_tr), sc.transform(Xd_te)                      # 같은 기준으로 변환
acc_scaled = accuracy_score(yd_te, KNN(5).fit(Xs_tr, yd_tr).predict(Xs_te))  # 스케일링 후
print(f"스케일링 없음: {acc_raw:.6f} · StandardScaler: {acc_scaled:.6f}")    # 원본 셀 11의 표


# ---------- 2-2 ----------
# 왜 · 스케일링 유무에 따른 점수 차이를 같은 분할로 비교하기 위해
pipe_knn = knn_pipe(5)                                                             # Scaler + KNN
acc_raw = accuracy_score(y_test, KNN(5).fit(X_train, y_train).predict(X_test))     # 그대로
acc_pipe = accuracy_score(y_test, pipe_knn.fit(X_train, y_train).predict(X_test))  # 스케일링
print(f"KNN (스케일링 없음): {acc_raw:.6f} · Pipeline (Scaler + KNN): {acc_pipe:.6f}")  # 원본 셀 13


# ---------- 2-3 ----------
# 왜 · 한 번의 분할 운이 아닌 20번의 분할로 스케일링 효과를 확인하기 위해
plain, piped = [], []                                                   # 분할마다 정확도
for seed in range(20):                                                  # split 20번
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)  # 새 분할
    plain.append(KNN(5).fit(Xa, ya).score(Xb, yb))                      # 그대로
    piped.append(knn_pipe(5).fit(Xa, ya).score(Xb, yb))                 # 스케일링
plain, piped = np.array(plain), np.array(piped)                         # 배열로
win, tie, lose = (plain > piped).sum(), (plain == piped).sum(), (piped > plain).sum()  # 세 경우
# 원본 셀 15
print(f"[split 20번] 스케일링 없음이 높음 {win} · 동점 {tie} · 스케일링이 높음 {lose}")
s_plain = cross_val_score(KNN(5), X, y, cv=cv).mean()                   # 5-Fold
s_pipe = cross_val_score(knn_pipe(5), X, y, cv=cv).mean()               # 5-Fold
print(f"[5-Fold] 스케일링 없음 {s_plain:.4f} · 스케일링 {s_pipe:.4f}")  # 순서가 뒤집힘

스케일링 없음: 0.650000 · StandardScaler: 0.983333
KNN (스케일링 없음): 1.000000 · Pipeline (Scaler + KNN): 0.933333


[split 20번] 스케일링 없음이 높음 8 · 동점 11 · 스케일링이 높음 1
[5-Fold] 스케일링 없음 0.9667 · 스케일링 0.9733


**실행 흐름**

1. 3-1 K별 점수
2. 3-2 K에 따른 경계

In [22]:
# ---------- 3-1 ----------
rows = []                                                               # K마다
# 왜 · K가 작으면 과적합, 크면 과소적합이 되는 흐름을 표로 보기 위해
for k in [1, 3, 5, 7, 9, 11, 13, 15]:                                   # 원본 셀 20의 후보
    pipe = knn_pipe(k).fit(X_train, y_train)                            # train에 fit
    tr = accuracy_score(y_train, pipe.predict(X_train))                 # 외운 문제
    cvs = cross_val_score(knn_pipe(k), X_train, y_train, cv=cv).mean()  # train 안 CV
    rows.append([k, tr, cvs, tr - cvs])                                 # 한 줄
k_df = pd.DataFrame(rows, columns=["K", "train 정확도", "CV 정확도", "격차 (train - CV)"])  # 표
print(k_df.round(4).to_string(index=False))                             # 원본 표


# ---------- 3-2 ----------
# 왜 · 특성 두 개만 써서 K에 따른 경계 변화를 그림으로 보기 위해
X_2d = X[:, 2:4]                    # petal length · width
# 원본 셀 22
X_tr, X_te, y_tr, y_te = train_test_split(X_2d, y, test_size=0.3, stratify=y, random_state=42)
acc_map = {}                        # K마다 test 정확도
for k in [1, 3, 5, 7, 9, 15]:       # 원본 드롭다운 후보
    acc_map[k] = round(knn_pipe(k).fit(X_tr, y_tr).score(X_te, y_te), 3)  # test 45개
print("K별 test 정확도:", acc_map)  # 경계 그림(3장)과 함께 읽기

 K  train 정확도  CV 정확도  격차 (train - CV)
 1     1.0000  0.9417           0.0583
 3     0.9583  0.9667          -0.0083
 5     0.9750  0.9583           0.0167
 7     0.9750  0.9333           0.0417
 9     0.9583  0.9583           0.0000
11     0.9583  0.9500           0.0083
13     0.9583  0.9500           0.0083
15     0.9667  0.9250           0.0417


K별 test 정확도: {1: 1.0, 3: 0.933, 5: 0.911, 7: 0.911, 9: 0.911, 15: 0.911}


**실행 흐름**

1. 4-1 특성 줄이기 비교
2. 4-2 permutation 중요도

In [23]:
from sklearn.inspection import permutation_importance  # 섞어서 성능 변화 보기


# ---------- 4-1 ----------
# 왜 · 특성을 줄여도 점수가 유지되는지 같은 CV로 비교하기 위해
s_all = cross_val_score(knn_pipe(5), X, y, cv=cv, scoring="accuracy")            # 4개
s_petal = cross_val_score(knn_pipe(5), X[:, 2:4], y, cv=cv, scoring="accuracy")  # petal 2개
print(f"petal 2개: {s_petal.mean():.4f} ± {s_petal.std():.4f}")                  # 원본 셀 25
print(f"전체 4개 : {s_all.mean():.4f} ± {s_all.std():.4f}")                      # 원본 셀 25
gap, sd = s_all.mean() - s_petal.mean(), min(s_all.std(), s_petal.std())         # 차이 · 흔들림
print(f"차이 {gap:.4f} vs 작은 쪽 표준편차 {sd:.4f}")                            # 근거 판단


# ---------- 4-2 ----------

# 왜 · KNN에는 내장 중요도가 없어 섞어 보기(permutation)로 특성 중요도를 재기 위해
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)  # 원본 셀 28
pipe_pi = knn_pipe(5).fit(Xa, ya)                                    # Scaler + KNN
imp = permutation_importance(pipe_pi, Xb, yb, n_repeats=30,          # test에서 30번씩 섞기
                             random_state=42, scoring="accuracy")
imp_df = pd.DataFrame({"피처": iris.feature_names,                   # 원본 표
                       "중요도(성능 하락폭)": imp.importances_mean, "± std": imp.importances_std})
imp_df = imp_df.sort_values("중요도(성능 하락폭)", ascending=False)  # 큰 순서
print(imp_df.round(4).to_string(index=False))                        # 원본 표

petal 2개: 0.9600 ± 0.0389
전체 4개 : 0.9733 ± 0.0249
차이 0.0133 vs 작은 쪽 표준편차 0.0249


               피처  중요도(성능 하락폭)  ± std
 petal width (cm)       0.2400 0.0442
petal length (cm)       0.2030 0.0576
sepal length (cm)       0.0067 0.0238
 sepal width (cm)      -0.0067 0.0288

**실행 흐름**

1. 5-1 dev 안에서 GridSearchCV
2. 5-2 동점 후보 확인

In [24]:
from sklearn.model_selection import GridSearchCV  # 후보 × CV


# ---------- 5-1 ----------

# 왜 · K와 투표 방식을 dev 안에서만 고르고 최종 test는 마지막에 한 번만 쓰기 위해
X_dev, X_final_test, y_dev, y_final_test = train_test_split(  # 수능 먼저 봉인 (원본 셀 31)
    X, y, test_size=0.2, stratify=y, random_state=42)
param_grid = {"knn__n_neighbors": [1, 3, 5, 7, 9, 11],                            # 6 × 2 = 12조합
              "knn__weights": ["uniform", "distance"]}
gs = GridSearchCV(knn_pipe(), param_grid, scoring="accuracy", cv=cv,              # dev 안에서만
                  return_train_score=True).fit(X_dev, y_dev)
print(f"탐색 범위 (dev) {len(X_dev)}개 · 봉인 (test) {len(X_final_test)}개")      # 120 · 30
print(f"CV 최고 조합 {gs.best_params_} · CV 최고 Accuracy {gs.best_score_:.4f}")  # 순서상 먼저
cv_table = pd.DataFrame(gs.cv_results_).rename(columns={"param_knn__n_neighbors": "K",  # 원본 셀 32
                                                        "param_knn__weights": "weights"})
for w in ["uniform", "distance"]:                                                 # weights별
    sub = cv_table[cv_table["weights"] == w].sort_values("K")                     # K 순
    print(w, "train:", sub["mean_train_score"].round(3).tolist())                 # 외운 문제
    print(w, "CV   :", sub["mean_test_score"].round(3).tolist())                  # 처음 보는 문제


# ---------- 5-2 ----------
# 왜 · 점수가 같은 후보가 여럿일 때 어떤 규칙으로 골랐는지 확인하기 위해
cols = ["K", "weights", "mean_test_score", "std_test_score"]  # 볼 열
ties = cv_table[cv_table["rank_test_score"] == 1][cols]       # rank 1인 행
print(f"공동 1위 조합 수: {len(ties)}개")                     # 원본 셀 35
print(ties.round(4).to_string(index=False))                   # 누구인지
print("best_params_ 가 고른 것:", gs.best_params_)            # 동점 중 순서상 먼저

탐색 범위 (dev) 120개 · 봉인 (test) 30개
CV 최고 조합 {'knn__n_neighbors': 3, 'knn__weights': 'uniform'} · CV 최고 Accuracy 0.9667
uniform train: [1.0, 0.969, 0.969, 0.967, 0.954, 0.958]
uniform CV   : [0.942, 0.967, 0.958, 0.933, 0.958, 0.95]
distance train: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
distance CV   : [0.942, 0.967, 0.958, 0.942, 0.958, 0.967]
공동 1위 조합 수: 3개
 K  weights  mean_test_score  std_test_score
 3  uniform           0.9667          0.0167
 3 distance           0.9667          0.0167
11 distance           0.9667          0.0167
best_params_ 가 고른 것: {'knn__n_neighbors': 3, 'knn__weights': 'uniform'}


**실행 흐름**

1. 6-1 차원의 저주 (거리 비율)
2. 6-2 쓸모없는 특성 수에 따른 점수

In [25]:
from scipy.spatial.distance import cdist  # 모든 쌍의 거리
from sklearn.datasets import make_blobs   # 군집이 뚜렷한 데이터


# ---------- 6-1 ----------


# 왜 · 차원이 커질수록 가장 가까운 점과 먼 점의 거리가 비슷해지는 현상을 숫자로 보기 위해
def min_max_ratio(Xr):                  # 점마다 (가장 가까운 거리 ÷ 가장 먼 거리)의 평균
    D = cdist(Xr, Xr)                   # 200 × 200 거리
    np.fill_diagonal(D, np.inf)         # 자기 자신 제외
    mn = D.min(axis=1)                  # 가장 가까운 거리
    np.fill_diagonal(D, 0)              # 최댓값 계산용
    return (mn / D.max(axis=1)).mean()  # 비율 평균


# 왜 · 데이터 분포에 따라 차원의 저주가 다르게 나타나는지 비교하기 위해
rng_cd = np.random.RandomState(42)                                      # 원본 셀 38
for name in ["균등난수", "정규분포", "군집 뚜렷"]:                      # 분포 3가지
    ratios = []                                                         # 차원마다
    for d in [2, 10, 50, 200]:                                          # 차원
        if name == "균등난수":                                          # 원본 셀 38
            Xr = rng_cd.rand(200, d)                                    # 0 ~ 1 균등
        elif name == "정규분포":                                        # 원본 셀 42 표
            Xr = rng_cd.randn(200, d)                                   # 표준정규
        else:                                                           # 구조가 있는 데이터
            Xr, _ = make_blobs(n_samples=200, n_features=d, centers=5,  # 5개 군집
                               cluster_std=0.5, random_state=42)
        ratios.append(round(float(min_max_ratio(Xr)), 3))               # 비율
    print(f"{name}: d=2 · 10 · 50 · 200 → {ratios}")                    # 한 줄


# ---------- 6-2 ----------
# 왜 · 쓸모없는 특성이 늘수록 KNN 점수가 떨어지는 것을 확인하기 위해
n_list = [0, 2, 5, 10, 20, 50, 100, 200]                               # 붙일 노이즈 수 (원본 셀 40)
rows = []                                                              # (seed, 노이즈 수, 정확도)
for seed in range(5):                                                  # seed 5개
    noise_pool = np.random.RandomState(seed).rand(len(X), max(n_list))  # 노이즈 200개를 한 번만
    cv_noise = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)  # seed마다 CV
    for n in n_list:                                                   # 앞에서부터 잘라 씀 (누적)
        X_aug = np.hstack([X, noise_pool[:, :n]])                      # 원래 4개 + 노이즈 n개
        rows.append([seed, n, cross_val_score(knn_pipe(5), X_aug, y, cv=cv_noise).mean()])  # CV
noise_df = pd.DataFrame(rows, columns=["seed", "노이즈 수", "acc"])    # 40행
noise_df = noise_df.groupby("노이즈 수")["acc"]                        # 노이즈 수별
print(noise_df.agg(평균="mean", 표준편차="std").round(4).to_string())  # 원본 표

균등난수: d=2 · 10 · 50 · 200 → [0.034, 0.347, 0.645, 0.805]
정규분포: d=2 · 10 · 50 · 200 → [0.037, 0.322, 0.614, 0.799]
군집 뚜렷: d=2 · 10 · 50 · 200 → [0.011, 0.039, 0.063, 0.073]


           평균    표준편차
노이즈 수                
0      0.9520  0.0073
2      0.8933  0.0200
5      0.8360  0.0252
10     0.8053  0.0242
20     0.7507  0.0361
50     0.6867  0.0641
100    0.6027  0.0289
200    0.5253  0.0292


**실행 흐름**

1. Iris test 1회
2. 펭귄 KNN 다시 설명
3. 펭귄 test 1회

In [26]:
# ---------- 7. 종합 실습 ----------
# 1단계: Iris test 1회 (dev에서 이미 고름)
# 왜 · dev에서 고른 설정 그대로 최종 test를 한 번만 채점하기 위해
final_test_acc = gs.best_estimator_.score(X_final_test, y_final_test)        # 처음이자 마지막
print(f"CV 최고 (dev 안) : {gs.best_score_:.4f}")                            # 고를 때 쓴 점수
print(f"test (수능)      : {final_test_acc:.4f}   (n={len(y_final_test)})")  # 보고
print(f"test에서 1명 틀릴 때 정확도 변화폭: {1 / len(y_final_test):.4f}")    # 해상도


# 2 · 3단계: 펭귄 KNN 0.7801 다시 설명 (파일별 2와 같은 데이터 · 분할)
# 왜 · 앞 권의 낮은 KNN 점수가 스케일링 부족 때문이었는지 같은 조건으로 확인하기 위해
meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]  # 측정값 4개
peng = pd.read_csv("../02_모델선택_펭귄/penguins.csv").dropna(subset=meas)  # 342마리
Xp, yp = peng[meas].to_numpy(), peng["species"].to_numpy()          # 특성 · 품종
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(                      # 파일별 2와 같은 분할
    Xp, yp, test_size=0.2, stratify=yp, random_state=7)
# 0.7801 재현
print(f"스케일링 없음 K=5 CV: {cross_val_score(KNN(5), Xp_tr, yp_tr, cv=5).mean():.4f}")
k_cv = {k: cross_val_score(knn_pipe(k), Xp_tr, yp_tr, cv=5).mean()  # 스케일링 + K마다 CV
        for k in [1, 3, 5, 7, 9, 11, 15]}
print("스케일링 + K별 CV:", {k: round(float(v), 4) for k, v in k_cv.items()})  # K 비교
best_k = max(k_cv, key=k_cv.get)                                    # 가장 높은 K (동점이면 먼저)


# 4단계: 고른 K로 펭귄 test 1회
# 왜 · 고른 K 그대로 test를 한 번만 써서 공정하게 비교하기 위해
peng_test = knn_pipe(best_k).fit(Xp_tr, yp_tr).score(Xp_te, yp_te)        # 처음이자 마지막
print(f"고른 K={best_k} · CV {k_cv[best_k]:.4f} · test {peng_test:.4f}")  # 보고

CV 최고 (dev 안) : 0.9667
test (수능)      : 0.9333   (n=30)
test에서 1명 틀릴 때 정확도 변화폭: 0.0333
스케일링 없음 K=5 CV: 0.7801
스케일링 + K별 CV: {1: 0.9818, 3: 0.9818, 5: 0.9854, 7: 0.9854, 9: 0.9818, 11: 0.9818, 15: 0.9818}
고른 K=5 · CV 0.9854 · test 1.0000
